Create a SQLite DB to practice.

DB contains 5 tables:

1. patients table (root)
2. encounters table (represents a single visit or interaction with all diagnoses, medication, and lab results hang off this table rather than directly off the patient table)
3. diagnoses table (connected to encounters table)
4. medications table (connected to encounters table)
5. lab_results table (connected to encounters table)

UNDERSTAND

PRAGMA foreign_keys = ON - SQLite doesn't enforce foreign keys by default. You have to enable it EACH CONNECTION. Knowing this shows understanding SQLite quirks vs. PostgreSQL

Insert Sample Data into the SQLite DB tables

In [4]:
import sys
import os

# project root folder
sys.path.append(os.path.join(os.getcwd(), '..'))

from build_database import build_ehr_database
build_ehr_database()
print("Ready.")

EHR database built successfully at: ehr.db
Tables created: patients, encounters, diagnoses, medications, lab_results
Ready.


Verify data was entered into created DB with a few quick queries

In [3]:
# Confirm row counts
for table in ["patients", "encounters", "diagnoses", "medications", "lab_results"]:
    count = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {table}", conn).iloc[0,0]
    print(f"  {table}: {count} rows")

# Preview a joined view — patient name + their encounters
query = """
    SELECT
        p.first_name || ' ' || p.last_name AS patient,
        e.encounter_date,
        e.encounter_type,
        e.provider
    FROM encounters e
    JOIN patients p ON p.patient_id = e.patient_id
    ORDER BY e.encounter_date
"""
df = pd.read_sql_query(query, conn)
print("\n", df.to_string(index=False))

  patients: 5 rows
  encounters: 9 rows
  diagnoses: 11 rows
  medications: 9 rows
  lab_results: 10 rows

      patient encounter_date encounter_type  provider
Maria Santos     2023-01-10     outpatient Dr. Patel
James Okafor     2023-02-20      inpatient   Dr. Kim
Priya Sharma     2023-03-12     outpatient Dr. Reyes
 Robert Chen     2023-04-07      emergency Dr. Smith
Lisa Trevino     2023-05-22     outpatient Dr. Patel
Maria Santos     2023-06-15     outpatient Dr. Patel
James Okafor     2023-08-05     outpatient   Dr. Kim
Priya Sharma     2023-09-18     telehealth Dr. Reyes
Lisa Trevino     2023-11-03     outpatient Dr. Patel


UNDERSTAND:

CHECK constraints on categorical fields - 'sex', 'encounter_type', 'flag' are all constrained to known values. In production EHR data, these get violated CONSTANTLY (this is where QC skills from EDA come in).

LOINC codes on lab results - 'loinc_code' is a real-world standard for lab test identifiers across health systems. (Mentioning LOINC and ICD-10 in an interview signals domain fluency)

**EXPLORATORY DATA ANALYSIS**

Task: Find all patients with a Type 2 diabetes diagnosis who had an HbA1c result above 7.0% in 2023

1. Find all diabetic ENCOUNTERS
   ICD-10 codes for Type 2 diabetes all start with 'E11'. Use "LIKE' to catch every subcode:

NOTE*: real EHR systems handle diagnoses using one of two common patterns:
1. Encounter-linked (how we built our DB): a diagnosis is a clinical event tied to a visit. This is how  EPIC's raw data and claims data work. If you want to know "does this patient have diabetes as an ongoing condition", you have to query across ALL their encounters and find at least one E11.x code, which is exactly what "LIKE E11%" across JOINs does.
2. Problem list / chronic condition list: a separate table that tracks ACTIVE, ONGOING conditions independent of individual encounters. This kind of table would 'live' at the patient level and it is queried directly. This is closer to how OMOP CDM's 'condition_era' table works.
   

In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("ehr.db")

query = """
    SELECT
        p.patient_id,
        p.first_name || ' ' || p.last_name AS patient,
        e.encounter_date,
        d.icd10_code,
        d.description
    FROM diagnoses d
    JOIN encounters e ON e.encounter_id = d.encounter_id
    JOIN patients p ON p.patient_id = e.patient_id
    WHERE d.icd10_code LIKE 'E11%'
    ORDER BY e.encounter_date
"""

df = pd.read_sql_query(query, conn)
print(df.to_string(index=False))

 patient_id      patient encounter_date icd10_code                        description
          1 Maria Santos     2023-01-10      E11.9      Type 2 diabetes, uncontrolled
          1 Maria Santos     2023-06-15     E11.65 Type 2 diabetes with hyperglycemia
          1 Maria Santos     2023-06-15      E11.9          Type 2 diabetes follow-up
          5 Lisa Trevino     2023-11-03      E11.9          Type 2 diabetes screening


Find all elevated HbA1c' results

In [6]:
query = """
    SELECT
        p.patient_id,
        p.first_name || ' ' || p.last_name AS patient,
        l.result_date,
        l.result_value AS hba1c_pct,
        l.flag
    FROM lab_results l
    JOIN encounters e ON e.encounter_id = l.encounter_id
    JOIN patients p   ON p.patient_id   = e.patient_id
    WHERE l.test_name = 'HbA1c'
      AND l.result_value > 7.0
      AND l.result_date BETWEEN '2023-01-01' AND '2023-12-31'
    ORDER BY l.result_value DESC
"""

df = pd.read_sql_query(query, conn)
print(df.to_string(index=False))

 patient_id      patient result_date  hba1c_pct flag
          1 Maria Santos  2023-01-10        8.2 high
          2 James Okafor  2023-08-05        7.9 high
          1 Maria Santos  2023-06-15        7.6 high


The full cohort query: combine both conditions.

A patient must satisfy both criteria: diabetes diagnosis AND elevated HbA1c

In [7]:
query = """
    SELECT DISTINCT
        p.patient_id,
        p.first_name || ' ' || p.last_name  AS patient,
        p.date_of_birth,
        p.sex,
        MAX(l.result_value)                  AS max_hba1c,
        COUNT(DISTINCT e.encounter_id)       AS total_encounters
    FROM patients p
    JOIN encounters e  ON e.patient_id   = p.patient_id
    JOIN diagnoses d   ON d.encounter_id = e.encounter_id
    JOIN lab_results l ON l.encounter_id = e.encounter_id
    WHERE d.icd10_code LIKE 'E11%'
      AND l.test_name     = 'HbA1c'
      AND l.result_value  > 7.0
      AND l.result_date BETWEEN '2023-01-01' AND '2023-12-31'
    GROUP BY p.patient_id, p.first_name, p.last_name,
             p.date_of_birth, p.sex
    ORDER BY max_hba1c DESC
"""

cohort = pd.read_sql_query(query, conn)
print(cohort.to_string(index=False))
print(f"\nCohort size: {len(cohort)} patients")

 patient_id      patient date_of_birth sex  max_hba1c  total_encounters
          1 Maria Santos    1968-03-14   F        8.2                 2

Cohort size: 1 patients


In real analysis, you often want the latest result, not the max. This requires a subquery or window function.

"ROW_NUMBER() OVER (PARTITION BY [...] ORDER BY [...])" pattern is one of the most common window functions in healthcare analytics. It comes up constantly for "most recent lab", "first diagnosis date", "last visit before discharge".

In [8]:
query = """
    WITH ranked_labs AS (
        SELECT
            e.patient_id,
            l.result_value  AS hba1c,
            l.result_date,
            l.flag,
            ROW_NUMBER() OVER (
                PARTITION BY e.patient_id
                ORDER BY l.result_date DESC
            ) AS rn
        FROM lab_results l
        JOIN encounters e ON e.encounter_id = l.encounter_id
        WHERE l.test_name = 'HbA1c'
    )
    SELECT
        p.patient_id,
        p.first_name || ' ' || p.last_name AS patient,
        r.result_date                       AS latest_lab_date,
        r.hba1c                             AS latest_hba1c,
        r.flag
    FROM ranked_labs r
    JOIN patients p ON p.patient_id = r.patient_id
    WHERE r.rn = 1
      AND r.hba1c > 7.0
    ORDER BY r.hba1c DESC
"""

df = pd.read_sql_query(query, conn)
print(df.to_string(index=False))

 patient_id      patient latest_lab_date  latest_hba1c flag
          2 James Okafor      2023-08-05           7.9 high
          1 Maria Santos      2023-06-15           7.6 high


Add a comorbidity flag

Real cohort queries often enrich with additional clinical context. Here we add a hypertension flag - patients with both diabetes AND hypertension are higher-risk.

The "LEFT JOIN" with "CASE WHEN [...] IS NOT NULL" pattern is the standard way to build comorbidity flags without dropping patients who don't have the condition.

In [9]:
query = """
    WITH diabetic_patients AS (
        SELECT DISTINCT e.patient_id
        FROM diagnoses d
        JOIN encounters e ON e.encounter_id = d.encounter_id
        WHERE d.icd10_code LIKE 'E11%'
    ),
    elevated_a1c AS (
        SELECT DISTINCT e.patient_id
        FROM lab_results l
        JOIN encounters e ON e.encounter_id = l.encounter_id
        WHERE l.test_name    = 'HbA1c'
          AND l.result_value > 7.0
    ),
    hypertensive AS (
        SELECT DISTINCT e.patient_id
        FROM diagnoses d
        JOIN encounters e ON e.encounter_id = d.encounter_id
        WHERE d.icd10_code LIKE 'I10%'
    )
    SELECT
        p.patient_id,
        p.first_name || ' ' || p.last_name AS patient,
        p.sex,
        CASE WHEN h.patient_id IS NOT NULL
             THEN 'yes' ELSE 'no' END       AS has_hypertension
    FROM patients p
    JOIN diabetic_patients dp ON dp.patient_id = p.patient_id
    JOIN elevated_a1c      ea ON ea.patient_id = p.patient_id
    LEFT JOIN hypertensive  h ON  h.patient_id = p.patient_id
    ORDER BY has_hypertension DESC
"""

df = pd.read_sql_query(query, conn)
print(df.to_string(index=False))

 patient_id      patient sex has_hypertension
          1 Maria Santos   F              yes


Query Challenge #1 from Claude using our faux patient DB:

Retreive the full name, date of birth, and sex of all female patients, sorted alphabetically by last name.

Tables needed: patients

Columns to return
Full name as a single column called "patient" (|| ' ' ||) is the concatenation string
"date_of_birth"
"sex"

In [13]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("ehr.db")

query = """
    SELECT
        p.first_name || ' ' || p.last_name AS patient,
        p.date_of_birth,
        p.sex
    FROM patients p
    WHERE p.sex = 'F'
    ORDER BY p.last_name
"""

df = pd.read_sql_query(query, conn)
print(df.to_string(index=False))

     patient date_of_birth sex
Maria Santos    1968-03-14   F
Priya Sharma    1979-07-22   F
Lisa Trevino    1990-05-05   F


Query Challenge #2 - single table aggregation

How many encounters of each "encounter_type" occurred in 2023? Return the type and count, sorted from most to least frequent

Tables needed: encounters

Columns to return:
"encounter_type"
a count column called "total"

In [18]:
conn = sqlite3.connect("ehr.db")

query = """
    SELECT
        e.encounter_type,        
        COUNT(*) AS total
    FROM encounters e
    GROUP BY e.encounter_type
    ORDER BY total DESC
"""
# "COUNT(*) AS total" is the standard pattern for 'how many rows per group' of encounter_types and 
# "GROUP BY" is splitting the counts into the correct encounter_type

df = pd.read_sql_query(query, conn)
print(df.to_string(index=False))

encounter_type  total
    outpatient      6
    telehealth      1
     inpatient      1
     emergency      1


Challenge #3: JOIN across two tables

List every patient's full name alongside the date and type of EACH of their encounters. Only include patients who have had at least one encounter.

Tables you'll need: patients, encounters
Columns to return:
patient's first and last name as 'patient'
'encounter_date'
'encounter_type'

Sorted by patient name, then by encounter date within each patient

In [20]:
conn = sqlite3.connect("ehr.db")

query = """
    SELECT
        p.first_name ||' '|| p.last_name AS patient,
        e.encounter_date,
        e.encounter_type
    FROM patients p
    JOIN encounters e ON e.patient_id = p.patient_id
    ORDER BY patient, encounter_date
"""
# the JOIN handles the "patients who have had at least one encounter. zero-encounter patients are excluded automatically

df = pd.read_sql_query(query, conn)
print(df.to_string(index=False))

     patient encounter_date encounter_type
James Okafor     2023-02-20      inpatient
James Okafor     2023-08-05     outpatient
Lisa Trevino     2023-05-22     outpatient
Lisa Trevino     2023-11-03     outpatient
Maria Santos     2023-01-10     outpatient
Maria Santos     2023-06-15     outpatient
Priya Sharma     2023-03-12     outpatient
Priya Sharma     2023-09-18     telehealth
 Robert Chen     2023-04-07      emergency


Challenge #4 - Three-table JOIN

Show each patient's full name, the encounter date, and any medications they were prescribed at that encounter. Include encounters even if no medications were prescribed - show those as blank.

Tables you'll need: patients, encounters, medications

Columns to return:
Full name as 'patient'
encounter_date
drug_name (blank if none)

Sorted by patient name, then encounter date

In [24]:
conn = sqlite3.connect("ehr.db")

query = """
    SELECT
        p.first_name ||' '|| p.last_name AS patient,
        e.encounter_date,
        CASE WHEN m.drug_name IS NOT NULL
             THEN m.drug_name ELSE 'none' END AS drug_name
    FROM patients p
    JOIN encounters e ON e.patient_id = p.patient_id
    LEFT JOIN medications m ON m.encounter_id = e.encounter_id
    ORDER BY patient, encounter_date
"""
# LEFT JOIN: keep every row from the left table (encounters in this case) regardless of whether a match
# exists in the right table (medications).
# Where there's no match, all columns from the right table come back as NULL

df = pd.read_sql_query(query, conn)
print(df.to_string(index=False))

     patient encounter_date    drug_name
James Okafor     2023-02-20      Insulin
James Okafor     2023-08-05    Metformin
Lisa Trevino     2023-05-22  Amoxicillin
Lisa Trevino     2023-11-03         none
Maria Santos     2023-01-10   Lisinopril
Maria Santos     2023-01-10    Metformin
Maria Santos     2023-06-15    Metformin
Priya Sharma     2023-03-12   Amlodipine
Priya Sharma     2023-09-18         none
 Robert Chen     2023-04-07      Aspirin
 Robert Chen     2023-04-07 Atorvastatin


Challenge #5 - Aggregation with a JOIN

For each patient, show their full name and the number of distinct lab tests they have had recorded. Include patients who have had no lab results - show those as 0

Tables you'll need: patients, encounters, lab_results

Columns to return:
Full name patients
'total_labs' (count of lab result rows)

In [33]:
conn = sqlite3.connect("ehr.db")

query = """
    SELECT
        p.first_name ||' '|| p.last_name AS patient,
        COUNT(l.result_id) AS total_labs
    FROM patients p
    JOIN encounters e ON e.patient_id = p.patient_id
    LEFT JOIN lab_results l ON l.encounter_id = e.encounter_id
    GROUP BY p.patient_id, p.first_name, p.last_name
    ORDER BY total_labs DESC
"""
# CASE WHEN l.result_value IS NOT NULL
             # THEN 0 ELSE 'none' END AS total_labs
# The CASE WHEN is not necessary here. Using COUNT on a NULL column naturally returns 0.

# Don't use COUNT(*) here. LEFT JOIN allows NULLS and COUNT(*) would count every row, including
# the null, which would give patients with no labs a count of 1 instead of 0. Counting a specific 
# column from the joined table skips nulls automatically.

df = pd.read_sql_query(query, conn)
print(df.to_string(index=False))

     patient  total_labs
Maria Santos           4
James Okafor           2
 Robert Chen           2
Priya Sharma           1
Lisa Trevino           1


Challenge #6:Subquery / CTE

Find all patients who have has at least two separate encounters where a 'critical' lab result was recorded.

Tables you'll need: patients, encounters, lab_results
Columns to return:
patient (full name)
critical_encounters (count of encounters with a critical result)

Sorted by 'critical_encounters' descending

HINT: A CTE will make this much cleaner than a nested subquery. Think about what you need to establisth first before you can count.

*CTE: Common Table Expression. A way to write a temporary named query that can be referenced later in the same SQL statement. It's like creating a mini table that only exists for the duration of your query.

WITH cte_name AS (
    SELECT ...
    FROM ...
    WHERE ...
)
SELECT *
FROM cte_name

In [45]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("ehr.db")

query = """
WITH critical_encounters AS (
    SELECT 
        e.patient_id,
        e.encounter_id
    FROM encounters e
    JOIN lab_results l ON l.encounter_id = e.encounter_id
    WHERE l.flag = 'critical'
)

SELECT 
    p.first_name ||' '|| p.last_name AS patient,
    COUNT(ce.encounter_id) AS critical_encounters
FROM patients p
JOIN critical_encounters ce ON ce.patient_id = p.patient_id
GROUP BY p.patient_id, p.first_name, p.last_name
HAVING COUNT(ce.encounter_id) >= 2
ORDER BY critical_encounters DESC
"""

# 'SELECT DISTINCT' collapses each patient down to one row which is useful for things like "did this patient ever have
# a critical result" but not great for "how many encounters had a critical result?".

# 'ORDER BY' needs a column to order by

df = pd.read_sql_query(query, conn)
print(df.to_string(index = False))

Empty DataFrame
Columns: [patient, critical_encounters]
Index: []


FINAL CHALLENGE - The Full Stack

Clinical Question: For each provider, show how many unique patients they saw, how many total encounters they conducted, and the average HbA1c result across all their patients who had one. Only include providers who saw at least 2 unique patients. Round the average to one decimal place.

Tables you'll need: patients, encounters, lab_results

Columns to return:
provider
unique_patients
total_encounters
avg_hba1c

Sorted by 'unique_patients' descending

Patterns needed from previous challenges:
JOIN and LEFT_JOIN (includes NULLs)
COUNT and GROUP BY
HAVING
ROUND(value, decimal_places)

Unique patient = SELECT DISTINCT ?
total_encounters = 
avg_hba1c = LEFT JOIN and AVERAGE and ROUND()

In [94]:
conn = sqlite3.connect("ehr.db")

query = """
SELECT 
    e.provider,
    COUNT(DISTINCT e.encounter_id) AS total_encounters,
    COUNT(DISTINCT e.patient_id) AS unique_patients,
    ROUND(AVG(CASE WHEN l.test_name ='HbA1c' THEN l.result_value END), 1) AS avg_hba1c
FROM encounters e
JOIN patients p ON p.patient_id = e.patient_id
LEFT JOIN lab_results l ON l.encounter_id = e.encounter_id
GROUP BY e.provider
HAVING COUNT(DISTINCT e.patient_id) >= 2
ORDER BY unique_patients DESC
"""

df = pd.read_sql_query(query, conn)
print(df.to_string(index = False))

 provider  total_encounters  unique_patients  avg_hba1c
Dr. Patel                 4                2        7.3


In [95]:
# providers

query = """
SELECT
    e.provider,
    e.encounter_id,
    l.test_name,
    l.result_value
FROM encounters e
LEFT JOIN lab_results l ON l.encounter_id = e.encounter_id
WHERE e.provider = 'Dr. Patel'
ORDER BY e.encounter_id
"""
df = pd.read_sql_query(query, conn)
print(df.to_string(index = False))

 provider  encounter_id  test_name  result_value
Dr. Patel             1 Creatinine           1.1
Dr. Patel             1      HbA1c           8.2
Dr. Patel             2      HbA1c           7.6
Dr. Patel             2       eGFR          72.0
Dr. Patel             8       None           NaN
Dr. Patel             9      HbA1c           6.1


In [69]:
# unique_patients

query = """
SELECT
    p.patient_id,
    e.provider,
    COUNT(DISTINCT e.patient_id) AS unique_patients
FROM patients p
JOIN encounters e ON e.patient_id = p.patient_id
GROUP BY e.provider
ORDER BY unique_patients DESC
"""

df = pd.read_sql_query(query, conn)
print(df.to_string(index=False))

 patient_id  provider  unique_patients
          1 Dr. Patel                2
          4 Dr. Smith                1
          3 Dr. Reyes                1
          2   Dr. Kim                1


In [97]:
# total encounters

query = """

SELECT 
    e.provider,
    COUNT(DISTINCT e.encounter_id) AS total_encounters
FROM encounters e
GROUP BY e.provider
"""

df = pd.read_sql_query(query, conn)
print(df.to_string(index=False))

 provider  total_encounters
  Dr. Kim                 2
Dr. Patel                 4
Dr. Reyes                 2
Dr. Smith                 1
